<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/Baseline_AttentionUNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import matplotlib.pyplot as plt

In [3]:
# Set seeds for reproducibility
np.random.seed(20)
tf.random.set_seed(20)

In [11]:
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_mask = np.load('/content/drive/MyDrive/Y_train_mask.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_mask = np.load('/content/drive/MyDrive/Y_test_mask.npy')

In [12]:
import numpy as np
import tensorflow as tf

# 1. Define the ESA class map
class_map = {
    10: ("Tree cover", "#006400"),
    20: ("Shrubland", "#ffbb22"),
    30: ("Grassland", "#ffff4c"),
    40: ("Cropland", "#f096ff"),
    50: ("Built-up", "#fa0000"),
    60: ("Bare / Sparse vegetation", "#b4b4b4"),
    70: ("Snow and ice", "#f0f0f0"),
    80: ("Permanent water bodies", "#0064ff"),
    90: ("Herbaceous wetland", "#0096a0"),
}

# 2. Identify which classes actually exist in your data
unique_labels = sorted(np.unique(Y_train_mask))
label_map = {old: new for new, old in enumerate(unique_labels)}

print("--- LABEL MAPPING TABLE ---")
for old_id, new_id in label_map.items():
    name = class_map.get(old_id, ("Unknown", ""))[0]
    print(f"Original ESA ID: {old_id:2} ({name:15}) -> New Neural Net ID: {new_id}")

# 3. Create the Look-Up Table (LUT) - THE FIX
# This maps the high ESA numbers to 0, 1, 2, 3, 4, 5 instantly across all pixels
lut = np.zeros(91, dtype=np.int32)
for old_id, new_id in label_map.items():
    lut[old_id] = new_id

# 4. Apply mapping to the whole 3D array (Spatial Mapping)
Y_train_ready = lut[Y_train_mask]
Y_test_ready = lut[Y_test_mask]

--- LABEL MAPPING TABLE ---
Original ESA ID: 10 (Tree cover     ) -> New Neural Net ID: 0
Original ESA ID: 20 (Shrubland      ) -> New Neural Net ID: 1
Original ESA ID: 30 (Grassland      ) -> New Neural Net ID: 2
Original ESA ID: 40 (Cropland       ) -> New Neural Net ID: 3
Original ESA ID: 50 (Built-up       ) -> New Neural Net ID: 4
Original ESA ID: 60 (Bare / Sparse vegetation) -> New Neural Net ID: 5
Original ESA ID: 80 (Permanent water bodies) -> New Neural Net ID: 6
Original ESA ID: 90 (Herbaceous wetland) -> New Neural Net ID: 7


In [13]:
# 5. One-Hot Encode for U-Net
num_classes = len(unique_labels)
Y_train_cat = tf.keras.utils.to_categorical(Y_train_ready, num_classes=num_classes)
Y_test_cat = tf.keras.utils.to_categorical(Y_test_ready, num_classes=num_classes)

print(f"X_train shape: {X_train.shape}") # no.of bands(7)
print(f"Y_train shape: {Y_train_cat.shape}") # no.of classes(8)

X_train shape: (639, 256, 256, 7)
Y_train shape: (639, 256, 256, 8)
